In [2]:
# This notebook assigns a canton to the start point and end point of each route.

# This cell imports all required libraries.

import pandas as pd
# pandas: works with tables (DataFrames).
# Used to load the scraped route data (result.csv) and to clean and merge tables.

import geopandas as gpd
# geopandas: extends pandas with a column for shapes (lines, areas, points).
# Used to read the shapefiles (route lines, canton areas) and to find
# the canton in which a point lies.

from shapely.geometry import Point
# shapely: the library geopandas uses to calculate shapes.
# Point represents a single location (east and north coordinate).
# Used to create the start point and end point of each route.

In [8]:
# Data loading: This cell loads the three data sources.

# Source 1: routes scraped from schweizmobil.ch
# One row per route with title, distance, elevation, difficulty, start and end point.
scraped_routes = pd.read_csv("result_wanderland.csv")

# Source 2: route geometries from the dataset "Wanderland Schweiz" (FEDRO / SchweizMobil)
# One row per route with the route number and the line of the route as coordinates.
coordinates = gpd.read_file("Geodaten/Route.shp")

# Source 3: canton boundaries from swissBOUNDARIES3D, edition 2026 (swisstopo)
# One row per canton with the canton number, name and the area of the canton as a shape.
cantons = gpd.read_file("Geodaten/Kantonsgrenzen/swissBOUNDARIES3D_1_5_TLM_KANTONSGEBIET.shp")

# Check: number of rows in each table
# 274 routes loaded, geodata available vor 359 routes and 26 cantons
print("Routes:", len(scraped_routes))
print("Wanderland:", len(coordinates))
print("Coordinates:", len(cantons))

Routes: 274
Wanderland: 359
Coordinates: 26


In [10]:
# Data loading: Shows the column names of the scraped routes
print(scraped_routes.columns.tolist())

['land', 'url', 'title', 'start', 'via', 'end', 'distance_km', 'el_gain_m', 'el_loss_m', 'el_max_m', 'el_min_m', 'duration_min', 'direction_note', 'asphalt_km', 'natural_km', 'singletrail_km', 'technical', 'condition', 'facts_raw', 'details_raw']


In [11]:
#  Data cleaning: This cell extracts the route number from the URL of each scraped route.
# Example: ".../wanderland/route-101" -> 101
# The route number is needed to link the scraped routes with the route geometries.

scraped_routes["route_nr"] = (
    scraped_routes["url"]
    .str.extract(r"route-(\d+)")[0]   # take the digits after "route-"
    .astype(int)                      # convert the text "101" into the number 101
)

# Check: URL and extracted route number side by side
scraped_routes[["url", "route_nr"]].head()

,url,route_nr
0,https://schweizmobil.ch/de/wanderland/route-101,101
1,https://schweizmobil.ch/de/wanderland/route-102,102
2,https://schweizmobil.ch/de/wanderland/route-103,103
3,https://schweizmobil.ch/de/wanderland/route-104,104
4,https://schweizmobil.ch/de/wanderland/route-105,105


In [14]:
# Data cleaning: This cell converts the route number in the route geometries from text to a number.
# In the shapefile, the route number (column "NrR") is stored as text, e.g. "101".
# To compare it with the route numbers of the scraped routes, it must be a number as well.
# A few internal entries have no valid route number (e.g. "SWL249.RT");
# errors="coerce" turns them into empty values instead of causing an error.

coordinates["route_nr"] = pd.to_numeric(coordinates["NrR"], errors="coerce")

# Check: how many entries could not be converted
print("Entries without a valid route number:", coordinates["route_nr"].isna().sum())

Entries without a valid route number: 4


In [16]:
# Data cleaning: This cell checks whether the two tables can be linked correctly before merging them.

# 1. How many scraped routes find a matching route number in the route geometries?
found = scraped_routes["route_nr"].isin(coordinates["route_nr"])
print("Scraped routes:", len(scraped_routes))
print("With matching geometry:", found.sum())

# 2. Do any of the scraped route numbers appear more than once in the route geometries?
relevant = coordinates[coordinates["route_nr"].isin(scraped_routes["route_nr"])]
print("Duplicate route numbers:", relevant["route_nr"].duplicated().sum())

Scraped routes: 274
With matching geometry: 274
Duplicate route numbers: 0


In [17]:
# merging: This cell links the scraped routes with the route geometries via the route number.

# Only the columns needed are taken from the route geometries:
# - geometry: the line of the route (used to find the start and end point)
# - AOrt, ZOrt: start and end location according to the geodataset (used for checking)
# how="left" keeps all scraped routes, even if no geometry were found.

routes_linked = scraped_routes.merge(
    coordinates[["route_nr", "AOrt", "ZOrt", "geometry"]],
    on="route_nr",
    how="left"
)

# Check: the number of rows must still be 274
print("Rows after linking:", len(routes_linked))

# Check: scraped start/end compared with start/end from the geodataset
routes_linked[["title", "start", "AOrt", "end", "ZOrt"]].head(10)

Rows after linking: 274


,title,start,AOrt,end,ZOrt
0,Sentier du Rhône,Genève,La Jonction,La Plaine,La Plaine
1,Far West du canton de Genève,Chancy,Chancy,Chancy,Chancy
2,Rive Droite,Meyrin,Meyrin La Tour,La Plaine,La Plaine
3,Au fil de la Versoix,Sauverny,Sauverny 102804,Versoix,121941
4,Entre Arve et Lac,Hermance,Hermance village,Choulex,Choulex village
5,Chemin des Gorges de Covatannaz,Ste-Croix,Ste-Croix,Baulmes,Baulmes
6,Tour de La Palette,Col du Pillon,Col du Pillon,Col du Pillon,Col du Pillon
7,Terrasses de Lavaux,St-Saphorin,"St-Saphorin, gare",Lutry,"Lutry, gare"
8,Sentier des Toblerones,Bassins,"Bassins, gare",Nyon,Nyon
9,Chemin de La Dôle,Saint-Cergue,St-Cergue,La Givrine,La Givrine


In [18]:
routes_linked

,land,url,title,start,via,end,distance_km,el_gain_m,el_loss_m,el_max_m,...,natural_km,singletrail_km,technical,condition,facts_raw,details_raw,route_nr,AOrt,ZOrt,geometry
0,wanderland,https://schweizmobil.ch/de/wanderland/route-101,Sentier du Rhône,Genève,NaN,La Plaine,20.0,460.0,480.0,432.0,...,NaN,NaN,leicht (Wanderweg),schwer,20 km | 460 m | 480 m | Höhenprofil | 432 m | ...,Höhenprofil ausblenden | : | 20 km | 460 m | 4...,101,La Jonction,La Plaine,"LINESTRING Z (2498944.875 1117403.284 374.377,..."
1,wanderland,https://schweizmobil.ch/de/wanderland/route-102,Far West du canton de Genève,Chancy,Vers Veaux–Bois de Chancy–Champlong,Chancy,11.0,300.0,300.0,426.0,...,NaN,NaN,leicht (Wanderweg),mittel,11 km | 300 m | 300 m | Höhenprofil | 426 m | ...,Höhenprofil ausblenden | : | 11 km | 300 m | 3...,102,Chancy,Chancy,"LINESTRING Z (2486537.811 1112036.129 358.678,..."
2,wanderland,https://schweizmobil.ch/de/wanderland/route-103,Rive Droite,Meyrin,NaN,La Plaine,19.0,380.0,460.0,506.0,...,NaN,NaN,leicht (Wanderweg),mittel,19 km | 380 m | 460 m | Höhenprofil | 506 m | ...,Höhenprofil ausblenden | : | 19 km | 380 m | 4...,103,Meyrin La Tour,La Plaine,"LINESTRING Z (2494242.385 1120756.25 426.85, 2..."
3,wanderland,https://schweizmobil.ch/de/wanderland/route-104,Au fil de la Versoix,Sauverny,"La Bâtie, Moulin-de-Richelien, Les Gravines",Versoix,10.0,120.0,200.0,462.0,...,NaN,NaN,leicht (Wanderweg),leicht,10 km | 120 m | 200 m | Höhenprofil | 462 m | ...,Höhenprofil ausblenden | : | 10 km | 120 m | 2...,104,Sauverny 102804,121941,"LINESTRING Z (2498459.953 1129749.361 457.794,..."
4,wanderland,https://schweizmobil.ch/de/wanderland/route-105,Entre Arve et Lac,Hermance,NaN,Choulex,18.0,240.0,190.0,474.0,...,NaN,NaN,leicht (Wanderweg),mittel,18 km | 240 m | 190 m | Höhenprofil | 474 m | ...,Höhenprofil ausblenden | : | 18 km | 240 m | 1...,105,Hermance village,Choulex village,"LINESTRING Z (2507947.316 1128462.873 379.385,..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
269,wanderland,https://schweizmobil.ch/de/wanderland/route-995,Fürstin-Gina-Weg Malbun,Malbun,NaN,Malbun,12.0,950.0,950.0,2359.0,...,NaN,NaN,mittel (Bergwanderweg),schwer,12 km | 950 m | 950 m | Höhenprofil | 2'359 m ...,Höhenprofil ausblenden | : | 12 km | 950 m | 9...,995,Malbun,Malbun,LINESTRING Z (2764638.671 1219146.844 1599.524...
270,wanderland,https://schweizmobil.ch/de/wanderland/route-996,Eschnerberg-Höhenweg,Schellenberg,NaN,Schellenberg,11.0,480.0,480.0,694.0,...,NaN,NaN,leicht (Wanderweg),mittel,11 km | 480 m | 480 m | Höhenprofil | 694 m | ...,Höhenprofil ausblenden | : | 11 km | 480 m | 4...,996,Schellenberg,Schellenberg,"LINESTRING Z (2759618.046 1233336.87 632.072, ..."
271,wanderland,https://schweizmobil.ch/de/wanderland/route-997,Ellhorn Rundweg,Balzers (Mäls),NaN,Balzers (Mäls),8.0,260.0,260.0,670.0,...,NaN,NaN,mittel (Bergwanderweg),leicht,8 km | 260 m | 260 m | Höhenprofil | 670 m | 4...,Höhenprofil ausblenden | : | 8 km | 260 m | 26...,997,Mäls,Mäls,"LINESTRING Z (2756265.418 1214266.974 474.859,..."
272,wanderland,https://schweizmobil.ch/de/wanderland/route-998,Triesenberger Philosophenweg,"Triesenberg, Rizlina",Masescha–Gaflei,"Triesenberg, Rizlina",11.0,620.0,620.0,1592.0,...,NaN,NaN,mittel (Bergwanderweg),mittel,11 km | 620 m | 620 m | Höhenprofil | 1'592 m ...,Höhenprofil ausblenden | : | 11 km | 620 m | 6...,998,Rizlina,Rizlina,"LINESTRING Z (2760433.855 1220952.22 1168.209,..."


In [19]:
# Controlling: This cell checks how the route lines are stored.
# LineString: one continuous line -> first point = start, last point = end
# MultiLineString: several separate line pieces -> order of the pieces is not guaranteed

print(routes_linked["geometry"].apply(lambda g: g.geom_type).value_counts())

geometry
LineString    274
Name: count, dtype: int64


In [20]:
# merging: This cell creates the start point and the end point of each route.
# Each route line consists of many points with coordinates (east, north, height).
# The first point of the line is the start, the last point is the end.
# [:2] keeps only east and north; the height is not needed to find the canton.

routes_linked["start_point"] = routes_linked["geometry"].apply(lambda line: Point(line.coords[0][:2]))
routes_linked["end_point"] = routes_linked["geometry"].apply(lambda line: Point(line.coords[-1][:2]))

# Check: first rows with start and end location and the new points
routes_linked[["start", "start_point", "end", "end_point"]].head()

,start,start_point,end,end_point
0,Genève,POINT (2498944.875 1117403.284),La Plaine,POINT (2488943.736 1115122.394)
1,Chancy,POINT (2486537.811 1112036.129),Chancy,POINT (2486537.811 1112036.129)
2,Meyrin,POINT (2494242.385 1120756.25),La Plaine,POINT (2488943.736 1115122.394)
3,Sauverny,POINT (2498459.953 1129749.361),Versoix,POINT (2501903.035 1126133.729)
4,Hermance,POINT (2507947.316 1128462.873),Choulex,POINT (2506219.275 1119786.43)


In [21]:
# Data cleaning: This cell adds the canton abbreviation to the canton boundaries.
# The boundaries only contain the official canton number (1-26) and the name in the local language.
# The canton number follows the official order of the Federal Constitution (1 = ZH, 2 = BE, ... 26 = JU).

canton_abbreviations = {
    1: "ZH", 2: "BE", 3: "LU", 4: "UR", 5: "SZ", 6: "OW", 7: "NW", 8: "GL", 9: "ZG",
    10: "FR", 11: "SO", 12: "BS", 13: "BL", 14: "SH", 15: "AR", 16: "AI", 17: "SG",
    18: "GR", 19: "AG", 20: "TG", 21: "TI", 22: "VD", 23: "VS", 24: "NE", 25: "GE", 26: "JU"
}

cantons["canton_abbr"] = cantons["KANTONSNUM"].map(canton_abbreviations)

# Check: canton number, name and abbreviation, sorted by number
cantons[["KANTONSNUM", "NAME", "canton_abbr"]].sort_values("KANTONSNUM")

,KANTONSNUM,NAME,canton_abbr
5,1,Zürich,ZH
22,2,Bern,BE
16,3,Luzern,LU
9,4,Uri,UR
4,5,Schwyz,SZ
6,6,Obwalden,OW
10,7,Nidwalden,NW
8,8,Glarus,GL
18,9,Zug,ZG
7,10,Fribourg,FR


In [25]:
# merging: This cell determines the canton of each start point.
# A spatial join (sjoin) links two tables by location instead of by a common column:
# each start point receives the canton abbreviation of the canton area it lies within.
# Points outside Switzerland (e.g. Liechtenstein) lie within no canton and are marked "abroad".

# 1. Turn the start points into a geo table with the same coordinate system as the cantons
start_points = gpd.GeoDataFrame(geometry=routes_linked["start_point"], crs=cantons.crs)

# 2. Spatial join: which canton area contains each point?
start_join = gpd.sjoin(start_points, cantons[["canton_abbr", "geometry"]], how="left", predicate="within")

# 3. Write the result into a new column; points without a canton are marked "abroad"
routes_linked["start_canton"] = start_join["canton_abbr"].fillna("abroad")

# Check: total number of start points, number abroad, and full list of all cantons
print("Total:", routes_linked["start_canton"].notna().sum())
print("Abroad:", (routes_linked["start_canton"] == "abroad").sum())
print(routes_linked["start_canton"].value_counts().to_string())

Total: 274
Abroad: 6
start_canton
GR        50
VS        32
BE        32
VD        25
TI        16
FR        14
SG        13
GL        11
SZ         9
JU         7
GE         6
SO         6
NW         6
UR         6
abroad     6
OW         5
SH         5
ZH         5
AR         5
NE         4
BL         4
TG         4
LU         2
AI         1


In [26]:
# This cell determines the canton of each end point.
# Same method as for the start points: spatial join of the end points with the canton areas.

# 1. Turn the end points into a geo table with the same coordinate system as the cantons
end_points = gpd.GeoDataFrame(geometry=routes_linked["end_point"], crs=cantons.crs)

# 2. Spatial join: which canton area contains each point?
end_join = gpd.sjoin(end_points, cantons[["canton_abbr", "geometry"]], how="left", predicate="within")

# 3. Write the result into a new column; points without a canton are marked "abroad"
routes_linked["end_canton"] = end_join["canton_abbr"].fillna("abroad")

# Check: total number of end points, number abroad, and full list of all cantons
print("Total:", routes_linked["end_canton"].notna().sum())
print("Abroad:", (routes_linked["end_canton"] == "abroad").sum())
print(routes_linked["end_canton"].value_counts().to_string())

Total: 274
Abroad: 7
end_canton
GR        50
BE        35
VS        32
VD        24
TI        16
FR        14
SG        13
GL        11
NW         9
SZ         9
abroad     7
JU         6
GE         5
SO         5
UR         5
SH         5
ZH         5
NE         4
BL         4
TG         4
AR         4
OW         3
LU         2
AI         2


In [28]:
# merging: This cell determines all cantons a route passes through.
# Instead of a point, the whole line of the route is used.
# predicate="intersects" finds every canton area the line touches or crosses.
# A route can pass through several cantons, so the join returns one row per route and canton;
# these rows are then combined into one text per route, e.g. "BE, NW, OW".

# 1. Turn the route lines into a geo table with the same coordinate system as the cantons
route_lines = gpd.GeoDataFrame(geometry=routes_linked["geometry"], crs=cantons.crs)

# 2. Spatial join: which canton areas does each line touch or cross?
line_join = gpd.sjoin(route_lines, cantons[["canton_abbr", "geometry"]], how="left", predicate="intersects")

# 3. For each route: all cantons the line touches, minus the start and end canton
#    -> only the cantons the route passes through in between remain
touched = line_join.groupby(level=0)["canton_abbr"].apply(lambda abbrs: set(abbrs.dropna()))

def cantons_in_between(row):
    in_between = touched[row.name] - {row["start_canton"], row["end_canton"]}
    return ", ".join(sorted(in_between)) if in_between else None

# 4. Write the result into a new column; routes without a canton in between stay empty
routes_linked["crossed_cantons"] = routes_linked.apply(cantons_in_between, axis=1)

# Check: routes passing through at least one canton in between
has_crossed = routes_linked["crossed_cantons"].notna()
print("Routes with cantons in between:", has_crossed.sum())
routes_linked.loc[has_crossed, ["title", "start_canton", "crossed_cantons", "end_canton"]]

Routes with cantons in between: 22


,title,start_canton,crossed_cantons,end_canton
6,Tour de La Palette,VD,BE,VD
63,Gastlosen Tour,FR,BE,FR
73,L’Intyamon de Montbovon à la Brâ,FR,VD,FR
76,Sentier du Creux du Van,NE,VD,NE
105,Schwarzwasser-Sense-Schluchtenweg,BE,FR,BE
116,Chemin du Val Terbi,JU,SO,JU
122,Wasserfallen-Rundweg,BL,SO,BL
124,Langenbruck-Bölcheflue–Rundweg,BL,SO,BL
126,Wolfsschlucht-Weg,SO,BE,SO
127,Tüfelsschlucht-Belchen-Weg,SO,BL,SO


In [29]:
# This cell creates the final table and saves it as a CSV file.

# Coordinates of start and end point as plain numbers (LV95: east "e" and north "n"),
# so they can be saved in a CSV file and used later for maps
routes_linked["start_e"] = routes_linked["start_point"].apply(lambda p: round(p.x))
routes_linked["start_n"] = routes_linked["start_point"].apply(lambda p: round(p.y))
routes_linked["end_e"] = routes_linked["end_point"].apply(lambda p: round(p.x))
routes_linked["end_n"] = routes_linked["end_point"].apply(lambda p: round(p.y))

# Final table: all original columns of the scraped routes + route number, cantons and coordinates
new_columns = ["route_nr", "start_canton", "end_canton", "crossed_cantons",
               "start_e", "start_n", "end_e", "end_n"]
original_columns = [col for col in scraped_routes.columns if col != "route_nr"]

routes_cantons = routes_linked[original_columns + new_columns]

# Save as a new file; the original scraped file stays unchanged
routes_cantons.to_csv("result_wanderland_cantons.csv", index=False)

# Check: number of rows and columns of the final table
print("Rows:", len(routes_cantons))
print("Columns:", routes_cantons.columns.tolist())

Rows: 274
Columns: ['land', 'url', 'title', 'start', 'via', 'end', 'distance_km', 'el_gain_m', 'el_loss_m', 'el_max_m', 'el_min_m', 'duration_min', 'direction_note', 'asphalt_km', 'natural_km', 'singletrail_km', 'technical', 'condition', 'facts_raw', 'details_raw', 'route_nr', 'start_canton', 'end_canton', 'crossed_cantons', 'start_e', 'start_n', 'end_e', 'end_n']


In [30]:
import os

# Search the whole project folder for files named ".gitignore"
for folder, subfolders, files in os.walk("."):
    if ".git" in folder.split(os.sep):
        continue                      # skip Git's internal folder
    if ".gitignore" in files:
        print(os.path.join(folder, ".gitignore"))

.\.idea\.gitignore
.\.idea.alt\.idea\.gitignore
